In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.neural_network import MLPClassifier
from sklearn.utils.class_weight import compute_sample_weight
import numpy as np

In [4]:
df = pd.read_csv("Datos_regresion.csv").drop("Diag.Egr1", axis=1)

# Separar características y target
X = df.drop(["Fallece"], axis=1)
y = df["Fallece"]

In [5]:
categorical_cols = ["Diag.Ing1", "Diag.Ing2", "Diag.Egr2"]
numerical_cols = ["Edad", "APACHE", "TiempoVAM"]

In [6]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist"), categorical_cols),
        ("num", RobustScaler(), numerical_cols)
    ]
)

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, stratify=y, random_state=42
)

In [8]:
from collections import Counter
from imblearn.over_sampling import SMOTE,ADASYN,BorderlineSMOTE, RandomOverSampler
print('Original dataset shape %s' % Counter(y))
sm = ADASYN(random_state=42)
X_res, y_res = sm.fit_resample(X_train, y_train)
print('Resampled dataset shape %s' % Counter(y_res))

Original dataset shape Counter({0: 176, 1: 32})
Resampled dataset shape Counter({0: 158, 1: 155})


In [9]:
print('Original dataset shape %s' % Counter(y))
sm = SMOTE(random_state=42)
X_smote, y_smote = sm.fit_resample(X_train, y_train)
print('Resampled dataset shape %s' % Counter(y_smote))

Original dataset shape Counter({0: 176, 1: 32})
Resampled dataset shape Counter({0: 158, 1: 158})


In [12]:
print('Original dataset shape %s' % Counter(y))
sm = BorderlineSMOTE(random_state=42)
X_bs, y_bs = sm.fit_resample(X_train, y_train)
print('Resampled dataset shape %s' % Counter(y_bs))

Original dataset shape Counter({0: 176, 1: 32})
Resampled dataset shape Counter({0: 158, 1: 158})


In [13]:
print('Original dataset shape %s' % Counter(y))
sm = RandomOverSampler(random_state=42)
X_ro, y_ro = sm.fit_resample(X_train, y_train)
print('Resampled dataset shape %s' % Counter(y_ro))

Original dataset shape Counter({0: 176, 1: 32})
Resampled dataset shape Counter({0: 158, 1: 158})


In [10]:
pipeline = ImbPipeline([
    ("preprocessor", preprocessor),
    ("mlp", MLPClassifier(hidden_layer_sizes=(200,150,100,50), 
                          activation="relu", 
                          solver="adam", 
                          early_stopping=True, 
                          max_iter=1000,
                          random_state=42))
])

In [15]:
pipeline.fit(X_res, y_res)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='infrequent_if_exist'),
                                                  ['Diag.Ing1', 'Diag.Ing2',
                                                   'Diag.Egr2']),
                                                 ('num', RobustScaler(),
                                                  ['Edad', 'APACHE',
                                                   'TiempoVAM'])])),
                ('mlp',
                 MLPClassifier(early_stopping=True,
                               hidden_layer_sizes=(200, 150, 100, 50),
                               max_iter=1000, random_state=42))])

In [16]:
y_pred = pipeline.predict(X_test)

In [17]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

print(classification_report(y_test, y_pred))
print(f"AUC-ROC: {roc_auc_score(y_test, y_pred):.2f}")

              precision    recall  f1-score   support

           0       1.00      0.72      0.84        18
           1       0.38      1.00      0.55         3

    accuracy                           0.76        21
   macro avg       0.69      0.86      0.69        21
weighted avg       0.91      0.76      0.80        21

AUC-ROC: 0.86


In [18]:
confusion_matrix(y_test, y_pred)

array([[13,  5],
       [ 0,  3]], dtype=int64)

# Comparando aumento de datos

In [19]:
pipeline.fit(X_smote, y_smote)
y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred))
print(f"AUC-ROC: {roc_auc_score(y_test, y_pred):.2f}")
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.93      0.72      0.81        18
           1       0.29      0.67      0.40         3

    accuracy                           0.71        21
   macro avg       0.61      0.69      0.61        21
weighted avg       0.84      0.71      0.75        21

AUC-ROC: 0.69
[[13  5]
 [ 1  2]]


In [20]:
pipeline.fit(X_bs, y_bs)
y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred))
print(f"AUC-ROC: {roc_auc_score(y_test, y_pred):.2f}")
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      0.67      0.80        18
           1       0.33      1.00      0.50         3

    accuracy                           0.71        21
   macro avg       0.67      0.83      0.65        21
weighted avg       0.90      0.71      0.76        21

AUC-ROC: 0.83
[[12  6]
 [ 0  3]]


In [21]:
pipeline.fit(X_ro, y_ro)
y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred))
print(f"AUC-ROC: {roc_auc_score(y_test, y_pred):.2f}")
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.93      0.78      0.85        18
           1       0.33      0.67      0.44         3

    accuracy                           0.76        21
   macro avg       0.63      0.72      0.65        21
weighted avg       0.85      0.76      0.79        21

AUC-ROC: 0.72
[[14  4]
 [ 1  2]]


# Prueba de campo

In [13]:
df = pd.read_csv("pc.csv", delimiter=";").drop(
    ["Pos.","Sexo","Piel","F.Ingreso UCI",
     "Fue un traslado","Procedencia","F.Egreso UCI",
     "Modos VAM"], axis=1)

# Separar características y target
X = df.drop("Fallece", axis=1)
X['TiempoVAM'] = X["TiempoVAM"]/24
y = df["Fallece"]

In [14]:
pipeline.predict_proba(pd.DataFrame(
    {
        'Edad':[60,69,78,47], 
        'Diag.Ing1':[16,8,14,3], 
        'Diag.Ing2':[21,14,16,0], 
        'Diag.Egr2':[17,0,12,0], 
        'APACHE':[16,16,23,9], 
        'TiempoVAM':[24,6,20,5]
    }
))

array([[0.20223495, 0.79776505],
       [0.05230091, 0.94769909],
       [0.16901137, 0.83098863],
       [0.98285039, 0.01714961]])

In [15]:
pipeline.predict_proba(X)
y_t = pipeline.predict(X)

In [16]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
confusion_matrix(y, y_t)

array([[59, 14],
       [ 0,  3]], dtype=int64)

In [17]:
print(classification_report(y, y_t))
print(f"AUC-ROC: {roc_auc_score(y, y_t):.2f}")

              precision    recall  f1-score   support

           0       1.00      0.81      0.89        73
           1       0.18      1.00      0.30         3

    accuracy                           0.82        76
   macro avg       0.59      0.90      0.60        76
weighted avg       0.97      0.82      0.87        76

AUC-ROC: 0.90


# Freezing

In [18]:
from joblib import dump

dump(pipeline, "new_workflow.joblib")

['new_workflow.joblib']

In [19]:
from joblib import load

model = load('new_workflow.joblib')

In [20]:
model.predict_proba(pd.DataFrame(
    {
        'Edad':[60,69,78,47], 
        'Diag.Ing1':[16,8,14,3], 
        'Diag.Ing2':[21,14,16,0], 
        'Diag.Egr2':[17,0,12,0], 
        'APACHE':[16,16,23,9], 
        'TiempoVAM':[24,6,20,5]
    }
))

array([[0.20223495, 0.79776505],
       [0.05230091, 0.94769909],
       [0.16901137, 0.83098863],
       [0.98285039, 0.01714961]])

In [332]:
X_res

,Edad,Diag.Ing1,Diag.Ing2,Diag.Egr2,APACHE,TiempoVAM
0,56,14,0,0,14,2
1,73,14,16,16,24,8
2,57,1,0,0,4,2
3,65,18,0,0,17,2
4,42,3,7,7,19,5
...,...,...,...,...,...,...
308,74,11,0,2,17,7
309,79,14,16,16,23,3
310,77,14,16,16,22,3
311,85,14,17,16,21,2


In [333]:
X

,Edad,Diag.Ing1,Diag.Ing2,Diag.Egr2,APACHE,TiempoVAM
0,47,3,0,0,9,5.0
1,35,31,0,0,12,4.0
2,71,15,16,0,15,4.0
3,59,7,0,0,4,1.0
4,69,14,16,16,25,1.0
...,...,...,...,...,...,...
71,61,31,0,43,14,5.0
72,53,40,16,17,15,34.0
73,21,15,16,8,17,3.0
74,59,7,3,0,18,8.0


In [334]:
pipeline.predict_proba(X)[y!=y_t]

array([[0.06191151, 0.93808849],
       [0.2343018 , 0.7656982 ],
       [0.0723382 , 0.9276618 ],
       [0.07378609, 0.92621391],
       [0.47739295, 0.52260705],
       [0.15935308, 0.84064692],
       [0.03677406, 0.96322594],
       [0.48205228, 0.51794772],
       [0.34278269, 0.65721731],
       [0.45779162, 0.54220838],
       [0.26812868, 0.73187132],
       [0.34040709, 0.65959291],
       [0.22566086, 0.77433914],
       [0.12974994, 0.87025006]])

In [280]:
X[y!=y_t]

,Edad,Diag.Ing1,Diag.Ing2,Diag.Egr2,APACHE,TiempoVAM
4,69,14,16,16,25,1.0
5,72,18,0,0,21,1.0
6,70,14,12,0,23,3.0
7,73,16,0,41,22,14.0
10,54,3,7,0,19,3.0
12,83,5,21,0,19,3.0
25,57,7,0,0,25,4.0
33,79,46,0,0,17,5.0
39,79,14,16,0,21,5.0
44,51,3,27,0,18,9.0
